In [1]:
%run Latex_macros.ipynb

<IPython.core.display.Latex object>

# Session 3: Learning from experience: Monte Carlo, TD and Q-learning

## Where this session fits

- **Series goal**
    - choose good actions by trial and error
    - from small tables to robots and language models
- **Position**
    - session 3: the last session computed values with a *known* model
    - now the model is unknown, so values are learned from sampled experience
    - the next session separates on-policy from off-policy and scales Q-learning with a neural network
- **This session's goal**
    - learn state and action values on Frozen Lake from episodes alone
- **By the end you can**
    - say why model-free learning needs many episodes and exploration
    - compute a Monte Carlo, a TD(0) and a Q-learning update by hand
    - read Q-learning on Frozen Lake
    - say what makes Q-learning *off-policy* (developed next session)


# Model-free Value methods

In the absence of a model
- we can only learn about the dynamics of the Environment
    - by *interacting* with the Environment
    - playing out episodes/trajectories (Monte Carlo waits for the end of each; TD does not)
    - gather rewards and successor state information

Each interaction yielding an episode/trajectory
- is a *sample*
- drawn from an unknown Environment Transition Probability distribution $\transp(\state', \rew \mid \state, \act)$

This means multiple episodes are necessary
- to draw a sufficiently diverse sample of $\transp(\state', \rew \mid \state, \act)$
    - for each $(\state , \act)$ combination
    
Contrast this with model-based methods
- all knowledge of the Environment is available without interacting

Notation: at step $\tt$ of an episode
- $\stateseq_\tt$, $\actseq_\tt$ and $\rewseq_{\tt+1}$ are the state, the action and the next reward (random, because an episode is a sample)
- $\rlret_\tt$ is the return from step $\tt$


# Monte Carlo (MC) method

The simplest model-free value method
- play an episode to the end
- update the value of *each visited state* toward the **return from that state**
    - the rewards actually received after it, discounted
$$
\statevalfun(\stateseq_\tt) \leftarrow \statevalfun(\stateseq_\tt) + \alpha \bigl( \rlret_\tt - \statevalfun(\stateseq_\tt) \bigr)
$$
- $\alpha$ is a learning rate: the estimate moves part of the way toward the target (explained with TD below)
- no bootstrapping: the target is a real sample of the return, not another estimate
- the update must wait until the episode has ended
- if a state appears twice in an episode, we update it with the return from each visit (every-visit MC; first-visit MC uses only the first)


# Temporal Difference (TD) methods

Temporal Difference (TD) is a family of value-based methods 
characterized by the use of
 *bootstrapping*
- creating an estimate that depends on other estimates

We will show two members of this family, the sampled versions of the last session's Bellman updates
- **TD(0)** learns $\statevalfun(\state)$: sampled policy evaluation (the policy evaluated is the one that generates the episodes)
- **Q-learning** learns $\actvalfun(\state, \act)$: sampled value iteration (the Bellman optimality update)

To be concrete: we focus on $\statevalfun(\state)$.

Model-free value learning replaces the Bellman *average* by one *sample*.

After a step from $\stateseq_\tt$ (reward $\rewseq_{\tt+1}$, next state $\stateseq_{\tt+1}$) form the **target**
$$
\rewseq_{\tt+1} + \disc \, \statevalfun_{k}(\stateseq_{\tt+1})
$$
- the immediate reward actually received
- plus the discounted *current estimate* of the successor's value
- $\statevalfun_k$ is the current estimate of a state's value: $k$ counts the successive estimates of that state's value

In contrast to model-based
- where the update averages over all actions and successors, because the probabilities are known

the model-free target uses the *single* action and successor that actually occurred.

TD(0) and Monte Carlo *evaluate* the policy that generates the episodes; Q-learning (below) also improves the policy.


The **temporal difference error** is the target minus the current estimate
$$
\delta = \rewseq_{\tt+1} + \disc \, \statevalfun_{k}(\stateseq_{\tt+1}) - \statevalfun_{k}(\stateseq_{\tt})
$$

Temporal differencing moves the estimate *toward the target*
$$
\statevalfun_{k+1}(\stateseq_{\tt}) = \statevalfun_{k}(\stateseq_{\tt}) + \alpha \, \delta
$$
- $k$ counts the successive estimates of $\statevalfun(\stateseq_\tt)$: $\statevalfun_k$ before the update, $\statevalfun_{k+1}$ after
- $\alpha$ is the learning rate
- the target is a single noisy sample, so we move only a fraction of the way (as in gradient descent)
- $\alpha = 1$ would jump all the way to the target

This is **TD(0)**, also called 1-step TD.

Each step updates only the state just left.


**p-step ahead updates**

The update above is based on the immediate (one step ahead) reward.

This method is called *1-step TD*

We can generalize this to *p-step TD* which uses
- rewards $\{ \rewseq_{\tt'} \; | \;  \tt \lt \tt' \le \tt+p \} $
- $\statevalfun_k (\stateseq_{\tt+p})$


$$
\delta = \left[ \sum_{i=1}^{p} \disc^{i-1} \rewseq_{\tt+i} + \disc^{p} \statevalfun_k(\stateseq_{\tt+p}) \right] - \statevalfun_k(\stateseq_{\tt})
$$



Note that
- the target uses the *current* estimate of the later states (iteration $k$), not a value updated after the step
- each step updates only one state (the one just left; for p-step TD, the one $p$ steps back)

This differs from the model-based sweeps of the last session
- they updated *every* state from the previous table
- TD updates only the states the episode actually visits


**Pseudo-code for p-step TD**

We give some code below.

The apparent complexity of the code arises
- because p-step TD, for $p \gt 1$
- involves returns that won't be experienced
- until $(p-1)$ steps after step $\tt$

So the update for $\statevalfun_k(\stateseq_\tt)$ is delayed $p-1$ steps
- until we accumulate the remaining rewards
- at the beginning of update $(k+1)$ we already have the value of $\statevalfun(\stateseq_{\tt+p})$ needed
    - i.e., $\statevalfun_k(\stateseq_{\tt+p})$ 

**Code**

- the code uses $p$ as the number of steps ahead, as on the slides
- it follows the Gymnasium interface (`reset()` returns `(state, info)`, `step()` returns five values)
- terminal states keep the value $0$; a time-out is treated like a terminal state

        from collections import defaultdict

        gamma = 0.9              # discount factor
        alpha = 0.5              # learning rate
        p = 3                    # number of steps ahead
        V = defaultdict(float)   # state-value function (terminal states stay at 0)

        def p_step_td_episode(env, policy, p):
            s, _ = env.reset()
            states, rewards = [s], []
            done = False

            while not done:
                a = policy(s)                          # policy(s) returns an action
                s, r, terminated, truncated, _ = env.step(a)
                done = terminated or truncated         # a time-out is treated like a terminal state (simplification)
                rewards.append(r)
                states.append(s)
                t = len(rewards)                       # steps taken so far

                if t >= p:                             # the state p steps back now has p rewards
                    u = t - p
                    G = sum(gamma**i * rewards[u + i] for i in range(p))
                    if not done:
                        G += gamma**p * V[states[t]]   # bootstrap from the current estimate
                    V[states[u]] += alpha * (G - V[states[u]])

            # end of episode: update the states that never got p rewards (terminal value is 0)
            T = len(rewards)
            for u in range(max(0, T - p + 1), T):
                G = sum(gamma**i * rewards[u + i] for i in range(T - u))
                V[states[u]] += alpha * (G - V[states[u]])


## TD versus Monte Carlo

Both learn values from episodes; they differ in the *target* of the update
- MC: the full return to the end of the episode (no bootstrapping)
- TD: the next reward plus the current estimate of the next state (bootstrapping)


To highlight the difference between TD and MC
- consider an episode $\state_1 \to \state_2 \to$ terminal
- the only reward is $+1$, earned on the last transition (from $\state_2$ into the terminal state)
- discount $\disc = 1$, learning rate $\alpha = 1$, all initial values $0$

What are the values after episode $1$, under MC? under TD(0)?


After episode $1$
- MC updates *both* visited states with the return of the whole episode: $\statevalfun(\state_1) = \statevalfun(\state_2) = 1$
- TD(0) updates $\state_1$ first, with target $0 + \statevalfun(\state_2) = 0$, then $\state_2$ with target $1 + 0 = 1$: $\statevalfun(\state_1) = 0$ and $\statevalfun(\state_2) = 1$
- a second episode lets TD(0) pass the information one step back: $\statevalfun(\state_1) = 1$

MC moves the reward along the whole path in one episode; TD(0) moves it one step per episode, but it does not wait for the episode to end.


A Monte Carlo update by hand, on (non-slippery) Frozen Lake
- one episode follows the route $0 \to 4 \to 8 \to 9 \to 13 \to 14 \to \mathbf{G}$ (reward $+1$ on the last move)
- each visited state is updated toward *its own* return, discounted: $\disc = 0.9$
- all values start at $0$ and $\alpha = 0.5$, so the new value is $0.5 \, \rlret_\tt$

What do squares $14$ and $0$ become?


| state | $0$ | $4$ | $8$ | $9$ | $13$ | $14$ |
|:--|:-:|:-:|:-:|:-:|:-:|:-:|
| return $\rlret_\tt$ | $0.59$ | $0.66$ | $0.73$ | $0.81$ | $0.90$ | $1.00$ |
| updated value | $0.30$ | $0.33$ | $0.36$ | $0.41$ | $0.45$ | $0.50$ |

- the return from square $0$ is $\disc^{5} = 0.9^5 \approx 0.59$: the reward comes five discounts later
- TD(0) on the same episode would update only square $14$, to $0.50$: the others have no information yet


| Method                  | Update Basis                                     | Backup Target                                                     |
|:----------------------- |:------------------------------------------------ |:----------------------------------------------------------------- |
| TD 1-step                  | After 1 step, immediate reward + next value      | $\rewseq_{\tt+1} + \disc\,\statevalfun(\stateseq_{\tt+1})$                                    |
| TD p-step | After p steps, sum of p rewards + value at $\tt+p$ | $\rewseq_{\tt+1} + \ldots + \disc^{p-1} \rewseq_{\tt+p} + \disc^{p} \statevalfun(\stateseq_{\tt+p})$ |
| MC ($p \to \infty$) | End of episode, full sample return               | $\rlret_\tt$ (total return to episode end)                               |

We summarize the comparison of TD and MC:

| Feature            | Monte Carlo                   | Temporal Difference           |
|:-------------------|:-----------------------------|:-----------------------------|
| Reliance           | Full episode return           | Single step + estimated value |
| Bootstrapping      | No                            | Yes                          |
| Update Timing      | End of episode                | At every step (after $p$ steps for p-step TD)                |



# Model-free vs Model-based methods: differences

Although the update equations for model-based and model-free methods
- appear similar on the surface
- there are deeper differences

The update equation for both model/model-free
- define $\statevalfun_{k+1}(\state)$ as a function of successor states $\statevalfun_{k}(\state')$

    

In model-based methods
- $\statevalfun_{k}(\state')$ is **not** a random variable
    - because the transition probabilities are *known constants*
- before convergence its error comes from *partial* information, not from randomness

In model-free methods, 
- the state $\state$, action $\act$, reward $\rew$, successor state $\state'$
- are *samples* from one step of a single actual episode/trajectory

$$\state = \stateseq_\tt, \act = \actseq_\tt,   \rew = \rewseq_{\tt+1}, \state' = \stateseq_{\tt+1}$$

- are *random*
    - dependent on the Environment's random response, drawn from $\transp(\state', \rew \mid \stateseq_\tt, \actseq_\tt)$, during the episode.

Thus
- $\statevalfun_{k+1}(\stateseq_\tt)$ is a random variable
- that depends on *another* random variable: $\statevalfun_{k}(\stateseq_{\tt+1})$

When 
- an estimate of a variable (e.g., $\statevalfun_{k+1}(\stateseq_\tt)$)
- is based on *another estimated value* (e.g., $\statevalfun_{k}(\stateseq_{\tt+1})$)

this is called *Bootstrapping*.

The *Temporal Difference (TD)* methods above bootstrap; Monte Carlo does not.

We can characterize a random estimate by
- its **bias**: how far its *average* is from the true value
- its **variance**: how much it fluctuates around its average
    - it depends on random variables, which have variance as well


## Converging to true $\statevalfun(\state)$

In order for a model-free method to converge to the true $\statevalfun(\state)$
- it must sample enough episodes
- to *experience* the full distribution of $\transp(\state', \rew \mid \state, \act)$

Note
- we don't actually *observe* $\transp(\state', \rew \mid \state, \act)$
- instead, each step gives a *random sample* $(\rewseq_{\tt+1}, \stateseq_{\tt+1})$ drawn from it
- by averaging many sampled *targets* for a state, over the actions the policy chooses and the successors the environment returns
- we converge on their expectation, which is the value
$$
\Exp{\pi}\left[ \rewseq_{\tt+1} + \disc \, \statevalfun_\pi(\stateseq_{\tt+1}) \mid \stateseq_\tt = \state \right] = \statevalfun_\pi(\state)
$$

The expectation is what the Bellman equation needs
- we get it by averaging samples, without ever knowing $\transp$
- convergence needs a learning rate that decays (the Frozen Lake run below uses a constant $\alpha$, which works only because that world is deterministic) and every state-action pair visited again and again


## Model-free vs Model-based methods: Summary

| Aspect                           | Model-based Bellman backup         | Model-free TD backup                                         |
|:-------------------------------- |:---------------------------------- |:------------------------------------------------------------ |
| Determinism/randomness           | Deterministic (no random sampling) | Stochastic (sampled; random variable)                        |
| Need for exploration             | No                                 | Yes                                                          |
| Error before convergence         | Yes, if values are not yet converged (it bootstraps from the previous table) | Yes: it also bootstraps, and each target is a single random sample |
| Variance of backup               | Zero (averaged over all outcomes)  | Nonzero (sample-specific, reduces with averaging)            |
| Source of update target          | Full probability model             | Single experience sample                                     |
| Convergence (with enough sweeps) | To true value                      | To expected value (with sufficient exploration and learning) |

What the comparison costs
- MC: its target is a real return, so on average it is right (*unbiased*), but it adds up many random rewards, so it fluctuates a lot (*high variance*)
- TD: its target contains an estimate, so it inherits that estimate's error (*biased*), but it depends on only one random step (*lower variance*)
- bias and variance are developed in a later session


## Greed is not (always) good

Consider the policy $\act = \pi(\state)$ used by a method during its learning phase.

Before learning
- each $\act$ will be equally likely
- symmetric initialization of initial, uninformed policy

If $\pi(\state)$ were completely deterministic
- in every episode during learning in which we arrive at state $\state$
- we would take the same action $\act$
- most likely the *first* action chosen in the first episode
- with all values $0$ at the start, a greedy agent breaks the tie the same way every time (say, always the first action: 'left', which just bumps into the wall)


But the action chosen in the first episode
- may not be the optimal one

A deterministic policy locks us into a potentially sub-optimal choice.

This is an illustration of the *exploration/exploitation* tradeoff 
- Choosing the action currently believed to be optimal is *exploitation* (*greedy* choice)
- Choosing a different action is called *exploration* (*non-greedy* choice)

In model-free methods
- we may be forced to make action choices *that are not optimal* with respect to the current policy
- in order to better sample the Environment

An $\epsilon$-greedy policy makes
- the greedy choice with probability $(1-\epsilon)$
- a random action with probability $\epsilon$ (which may also be the greedy one)

## Q-learning: Action-value function

Just as in the model-based method (the last session), Q-learning creates a function from state-action pairs to expected returns

$$
\actvalfun: \States \times \Actions \to \Reals
$$

The best action in state $\state$ is read off directly

$$
\pi(\state) = \argmax{\act} \actvalfun(\state, \act)
$$

- this is the *greedy* policy of the current $\actvalfun$; it is optimal when $\actvalfun = \actvalfun_*$
- no model is needed for this step
- with only $\statevalfun$, acting greedily would need the model (a one-step lookahead): that is why model-free control learns action values
- note: the $\argmax{}$ results in a *deterministic* policy just as before


The optimal action value satisfies the Bellman optimality equation
$$
\actvalfun_*(\state, \act) = \E\left[ \rewseq_{\tt+1} + \disc \max{\act'} \actvalfun_*(\stateseq_{\tt+1}, \act') \mid \stateseq_\tt = \state, \actseq_\tt = \act \right]
$$

Q-learning replaces the expectation by *one sample* and moves the estimate toward it
$$
\actvalfun_{k+1}(\stateseq_\tt, \actseq_\tt) = \actvalfun_{k}(\stateseq_\tt, \actseq_\tt) + \alpha \left[ \rewseq_{\tt+1} + \disc \max{\act'} \actvalfun_{k}(\stateseq_{\tt+1}, \act') - \actvalfun_{k}(\stateseq_\tt, \actseq_\tt) \right]
$$
- the bracket is a TD error, now for action values
- the target uses the best action in the next state, whichever action the agent actually takes next

We show some pseudo-code for Q-learning below.


A Q-learning update by hand, on Frozen Lake
- all values start at $0$; $\alpha = 0.9$, $\disc = 0.9$
- the agent moves from square $14$ into **G** and earns $+1$
    - target $= 1 + 0.9 \times 0 = 1$ (the next state is terminal)
    - $\actvalfun(14, \text{right}) = 0 + 0.9\,(1 - 0) = 0.9$
- in a later episode it moves from square $13$ to square $14$ (reward $0$)
    - target $= 0 + 0.9 \times \max{\act'} \actvalfun(14, \act') = 0.9 \times 0.9 = 0.81$
    - $\actvalfun(13, \text{right}) = 0 + 0.9\,(0.81 - 0) \approx 0.73$
- information moves back one square each time a pair is revisited: the episode-$70$ picture shows $0.90$ at square $14$


**Pseudo code for Q-learning**

    Initialize Q(s, a) arbitrarily for all states s and actions a (often Q(s, a) = 0)
    
    Set learning rate alpha, discount factor gamma, exploration rate epsilon
    
    For episode = 1 to number_of_episodes:
        Initialize state s
        
        Repeat until s is terminal:
            With probability epsilon:
                Choose a random action a (exploration)
            Otherwise:
                Choose action a = argmax_a Q(s, a) (exploitation)
                
            Take action a, observe reward r and next state s'
            
            Update Q(s, a) using:
                Q(s, a) = Q(s, a) + alpha * [r + gamma * max_{a'} Q(s', a') - Q(s, a)]
                (if s' is terminal, the max term is 0)
                
            s = s'


In the above code
- we build a table $Q$ for the action value function $\actvalfun$ (an estimate of $\actvalfun_*$) 
- You can see the $\epsilon$-greedy strategy

        With probability epsilon:
            Choose a random action a (exploration)
        Otherwise:
            Choose action a = argmax_a Q(s, a) (exploitation)
- Update assumes subsequent action choices are greedy  

        max_{a'} Q(s', a')
        
        
- Model-free: Notice that we don't make use of
    -  $\transp(\state', \rew | \state, \act)$
    - or any reward other than the one received by taking the chosen action

**Q-learning is off-policy**
- the *behavior* policy, which chooses the actions, is $\epsilon$-greedy
    - it explores with probability $\epsilon$ (a random action, which may also be the greedy one)
- the *target* policy, which the update learns about, is greedy
    - because the target uses $\max{\act'}$, *even if* the action actually taken next was not the max

Behavior and target policies differ: *off-policy*
- MC and TD(0) as presented evaluate the policy that generates the episodes (they are *on-policy*)
- off-policy is about *which policy is learned*, not about model-free versus model-based
- the next session develops this


### Q-learning in action: Frozen Lake

Here we see Q-learning in action, using the Frozen Lake environment.

We will show an animation of
- the Value table and policy
- as derived from the Q-table

Here is the value before Q-learning begins.

<br>
<table>
    <center><strong>Value function (uninitialized) as a table</strong></center>
<tr>
    <td><img src="images/Frozen_Lake_env.png"></td>
    <td><img src="images/qlearning_policy_evolution_0.png"></td>
</tr>
</table>

**The Frozen Lake run behind these pictures**

- the same grid as the last session, squares numbered $0$ to $15$; reward $+1$ on entering **G**, episodes end at **G** or a hole
- **non-slippery** (the real game slips; a later session returns to it), discount $\disc = 0.9$
- learning rate $\alpha = 0.9$ (a large step size, which works here because nothing is noisy)
- each episode is cut off after $100$ steps
- $\epsilon$-greedy, with $\epsilon = 0.01 + 0.99\,e^{-0.005 \cdot \text{episode}}$: about $0.4$ after $200$ episodes, about $0.02$ after $1000$
- $20{,}000$ episodes in total; the table starts at $0$
- picture legend: pale blue is the start, grey is frozen, black is a hole, green is the goal; each square shows its best action value ($\max{\act}\,\actvalfun$) and the arrow is the greedy action; the goal shows $0.00$ because it is terminal; the actions are numbered left, down, right, up


Here is an evolution in the early episodes of Q-learning training.

<br>
<table>
    <center><strong>Value function: Q-learning early episodes</strong></center>
<tr>
    <td><center><strong>Early episode </strong><br><br>It takes many episodes for the reward to diffuse from G<br>We need to experience an episode that reaches the neighbor of G</center></td>
    <td><center><strong>Slightly later episode</strong></center></td>
</tr>
    
<tr>
    <td><img src="images/qlearning_policy_evolution_70.png"></td>
    <td><img src="images/qlearning_policy_evolution_145.png"></td>
</tr>
</table>

Hotter (yellower) colors represent higher values; the color scale differs between the pictures.

In the left-most early episode
- we see the Goal state (G) reward information diffusing to its immediate neighbor (state 14)
- and the arrow (policy derived from Q-table)
    - moving from neighbor to G
    
In the slightly later episode
- we see the information from G diffusing farther away
- and policy being update based on the new information

**Animation** (it plays once, about 50 seconds, and does not replay without reloading the page)
- the table after each episode, from episode $0$ to $19{,}999$

<img src="images/q_table_evolution_refined.gif" width=45%>


**The table at the end of training** (episode $19{,}999$)

<img src="images/qlearning_policy_evolution_final.png" width=45%>

- the arrows are the greedy actions of the table; the start's value is $0.59$, the optimal value from the last session
- not fully converged: squares $1$ and $10$ do not point to the best action, square $6$ still reads $0.59$ (the optimum is $0.81$), and squares $2$ and $3$ were rarely visited, so they still read $0$


We can observe the **average reward per episode** (the success rate, smoothed over 100 episodes)
- as a function of the number of episodes
- it is *not* the discounted return of the start state ($0.9^5 = 0.59$): each episode earns $1$ or $0$
- it counts *training* episodes, exploratory moves included; it rises to about $1$ as $\epsilon$ decays (to $0.01$) and the table improves

<table>
    <center><strong>Q-learning: average reward per episode (success rate)</strong></center>
<tr>
    <td><img src="images/Frozen_Lake_Qlearning_return_vs_episode.png"></td>
</tr>
</table>
    

Compare with the last session
- value iteration: a handful of sweeps over all squares, but it needs the model
- Q-learning: thousands of episodes of experience (about $1000$ before the success rate saturates), but no model
- both end with the same value at the start square, $0.59$


# What we established, and what it sets up

- **Established**
    - without a model, values are learned from sampled episodes, which needs exploration
    - Monte Carlo updates toward the real return (it waits for the episode to end)
    - TD updates toward a *bootstrapped* target (reward plus the current estimate), after every step
    - Q-learning learns action values and reads the policy off them, with no model
- **Sets up**
    - next session: on-policy vs off-policy, SARSA, and deep Q-learning (a network replaces the table)
    - later: bias and variance of these estimators, and learning the policy directly


In [4]:
print("Done")

Done
